# 01. Train

Mỗi lượt được lưu trong `outputs/runs/<tên lượt>` và ghi lên W&B. Nếu Colab ngắt phiên, chạy lại notebook với cùng lượt thì quá trình sẽ tiếp tục từ checkpoint gần nhất.

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Đổi thành địa chỉ repo của bạn. Bỏ qua nếu đã giải nén repo vào /content hoặc thêm vào Kaggle.
REPO_URL = "https://github.com/TEN-TAI-KHOAN/vihsd-lora.git"

def find_repo():
    places = [Path.cwd(), *Path.cwd().parents, Path("/content/vihsd-lora"), Path("/kaggle/working/vihsd-lora")]
    places += list(Path("/kaggle/input").glob("*")) + list(Path("/kaggle/input").glob("*/vihsd-lora"))
    return next((place for place in places if (place / "src" / "vihsd_lora").is_dir()), None)

repo = find_repo()
if repo is None:
    repo = Path("/content" if Path("/content").exists() else "/kaggle/working") / "vihsd-lora"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
if not os.environ.get("VIHSD_SKIP_INSTALL"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
sys.path.insert(0, str(repo / "src"))
print("Repo:", repo)

In [ ]:
from vihsd_lora.config import Settings
from vihsd_lora.env import default_output_root, describe_gpu, detect_platform, load_secrets

print("Nền tảng:", detect_platform())
print("GPU:", describe_gpu())
print("Secret đã có:", load_secrets())

settings = Settings.from_env(output_root=default_output_root())
print("Kết quả lưu tại:", settings.output_root)

## Chọn lượt

| Lượt | Cách train | r | Epoch |
|---|---|---|---|
| A | LoRA | 8 | 10 |
| B | Full fine-tune | | 10 |
| C | LoRA | 2 | 10 |
| D | LoRA | 16 | 10 |
| E | LoRA | 8 | 30 |

Có thể chạy nhiều lượt liền nhau, hoặc chia mỗi thành viên một lượt nếu dùng chung một W&B project.

In [ ]:
RUNS_TO_TRAIN = ["A"]
SEED = 42
WANDB_TEAM = None  # Điền tên team W&B nếu cả nhóm dùng chung một không gian

settings.wandb_entity = WANDB_TEAM or settings.wandb_entity

In [ ]:
import gc
import pandas as pd
import torch
from vihsd_lora.train import train_run

summaries = []
for key in RUNS_TO_TRAIN:
    summaries.append(train_run(key, seed=SEED, settings=settings))
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

columns = ["slug", "trainable_params", "trainable_pct", "dev_accuracy", "dev_macro_f1", "best_epoch",
           "mean_epoch_seconds", "gpu_peak_mb", "saved_mb"]
display(pd.DataFrame(summaries)[columns])

## Xem biểu đồ

Mở link W&B in ra trong lúc train. Biểu đồ theo epoch nằm ở nhóm `by_epoch`, gồm `dev_macro_f1`, `dev_accuracy`, `train_loss`, `epoch_seconds` và `gpu_peak_mb`. Chọn nhiều lượt trong bảng Runs để so trên cùng một biểu đồ.